In [ ]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay


class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self):
        script = textwrap.dedent(f"""
    

        import numpy as np
        from artiq.experiment import *
        from artiq.language.core import delay, kernel
        from kexp import Base, img_types, cameras

        class hf_bec(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,save_data=True,
                                camera_select=cameras.andor,
                                imaging_type=img_types.ABSORPTION)
                
                # self.xvar('t_tof',np.linspace(20.,3000.,7)*1.e-6)
                self.p.t_tof = 1700.e-6

                

                # self.xvar('do_405_pulse',[0,1])
                self.p.do_405_pulse = 1
                # self.xvar('do_980_pulse',[0,1])
                self.p.do_980_pulse = 1
                self.p.amp_dds_405 = 0.06
            # 


                    # self.xvar('compress',[0,1])
                self.p.compress = 0


                self.xvar('frequency_eo_980', np.arange(410.,430.,0.1)*1.e6)

                self.p.frequency_eo_980 = 418.1e6

                # self.xvar('t_tweezer_paint_rampdown',np.linspace(0.0,10.,5)*1.e-3)

                # self.xvar('t_tweezer_hold', np.linspace(0.0, 1050.0, 5) * 1.e-3)
                self.p.t_tweezer_hold = 671.e-3

                # self.p.v_pd_hf_tweezer_1064_rampdown3_end=3.5

                self.p.hf_imaging_detuning = -568.e6

                self.p.amp_imaging = 0.1
                # self.xvar('v_pd_ry_980',np.linspace(0.,1.,5))
                self.p.v_pd_ry_405 = 0.6
                self.p.v_pd_ry_980 = 2.8

                self.p.i_hf_raman = 182.

                
                # self.xvar('beans',np.linspace(0,30,30))
                self.p.N_repeats = 1
                self.finish_prepare(shuffle=True)

                if self.p.do_405_pulse == 1:
                    print(f'doing 405 pulse')
                else:
                    print(f'not doing 405 pulse')
                if self.p.do_980_pulse == 1:
                    print(f'doing 980 pulse')
                else:
                    print(f'not doing 980 pulse')

            @kernel
            def scan_kernel(self):
                
                self.ry_405.set_power(self.p.v_pd_ry_405)
                self.ry_980.set_power(self.p.v_pd_ry_980)

                if self.p.compress:
                        self.p.t_tof = 450.e-6
                if self.p.do_980_pulse == 1:
                    self.ry_980.sweep_to(self.p.frequency_eo_980)

                # self.ry_980.set_power(9.9)

                self.set_imaging_detuning(frequency_detuned=self.p.hf_imaging_detuning)
                self.imaging.set_power(self.p.amp_imaging)

                if self.p.compress:
                    self.prepare_hf_tweezers(squeeze=True)
                else:
                    self.prepare_hf_tweezers(squeeze=False, do_tweezer_evap_3=True, do_tweezer_evap_2=True)

                # self.tweezer.ramp(t=self.p.t_tweezer_squeezer_ramp_1,
                #                         v_start=self.p.v_pd_hf_tweezer_1064_rampdown3_end,
                #                         v_end=self.p.v_pd_tweezer_squeeze_rampup_handoff_lp,
                #                         low_power=True, paint=False, keep_trap_frequency_constant=False,
                #                         cubic_ramp=self.cubic_ramp)


                if self.p.do_405_pulse == 1:
                    self.ry_405.reboot()
                    self.ry_405.dds_sw.set_dds(amplitude=self.p.amp_dds_405)
                    self.ry_405.on()
                if self.p.do_980_pulse == 1:
                    self.ry_980.on()
                


                delay(self.p.t_tweezer_hold)

                self.ry_405.off()
                self.ry_980.off()
                self.ry_405.ttl_shutter.off()

                delay(40e-3)

                self.tweezer.off()

                delay(self.p.t_tof)
                self.abs_image()

                self.outer_coil.off()

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                self.end(expt_filepath)

        """)
        return script

In [16]:
eBuilder = ExptBuilder()

In [17]:
df = 10.e6
for i in range(25):
    print(f'scan num {i}')
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt())
    eBuilder.run_expt()

scan num 0
0  200 values of frequency_eo_980. 200 total shots. 600 total images expected.
Run ID: 73533
doing 405 pulse
doing 980 pulse
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 818], 'phase': 0.0, 'dimension': 0, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 0 um, phase = 0.0 pi, x-center = 993, y-center = 818

 Run ID: 73533
shot 1/200 done
shot 2/200 done
shot 3/200 done
shot 4/200 done
shot 5/200 done
shot 6/200 done
shot 7/200 done
shot 8/200 done
shot 9/200 done
shot 10/200 done
shot 11/200 done
shot 12/200 done
shot 13/200 done
shot 14/200 done
shot 15/200 done
shot 16/200 done
shot 17/200 done
shot 18/200 done
shot 19/200 done
shot 20/200 done
shot 21/200 done
shot 22/200 done
shot 23/200 done
shot 24/200 done
shot 25/200 done
shot 26/200 done
shot 27/200 done
shot 28/200 done
shot 29/200 done
shot 30/200 done
shot 31/200 done
shot 32/200 done
shot 33/200 done
shot 34/200 done
shot 35/200 done
shot 3

In [18]:
# eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(10.e6,10.e6))

In [19]:
from kexp import EthernetRelay
from waxx.util.guis.als.als_gui_client import ALSGuiClient
from waxx.util.guis.precilaser.precilaser_gui_client import PrecilaserGuiClient

relay = EthernetRelay()
relay.source_off()

als = ALSGuiClient()
ok = als.run_shutdown_sequence()

precilaser = PrecilaserGuiClient()
ok = precilaser.run_shutdown_sequence()